# Full benchmarking timeline (entry-point notebook)

This notebook walks you through the entire kermit benchmarking pipeline end-to-end:

1. **Generate** a LUBM-1 dataset on demand (`bench gen lubm`).
2. **Run** the time and space benchmarks across both index structures.
3. **Load** the resulting reports into a pandas DataFrame.
4. **Plot** four inline `matplotlib` figures.

Run the cells top-to-bottom (`Run All`) to reproduce. The first invocation triggers a full release build (~3-5 min); subsequent runs reuse the cargo build but regenerate the LUBM data from scratch (~30-60s for phase 1).

## Prerequisites

1. **Inside `nix develop`** if on NixOS — sets `LD_LIBRARY_PATH` for `libstdc++` and `libz` so numpy/matplotlib wheels load.
2. **JDK 8 on PATH** — the flake provides `pkgs.jdk8`; on Ubuntu, `apt install openjdk-8-jre`.
3. **Kermit-lab venv populated**: from the repo root, `cd python/kermit-lab && uv sync` (one-time).
4. **Launch this notebook** with: `cd python/kermit-lab && uv run --with jupyter jupyter lab notebooks/`, then open `00_full_timeline.ipynb`.

This notebook is the entry point. The other seven notebooks under `notebooks/` (`01_quick_start`, `02_scaling`, ...) are topical deep-dives that assume `bench-runs/` and `target/criterion/` are already populated by an earlier run.

In [ ]:
import os, subprocess, shutil, time
from pathlib import Path

# Resolve the repo root by walking up until we find Cargo.toml.
# This makes the notebook robust to wherever the kernel was launched from.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "Cargo.toml").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("Could not locate Kermit repo root (no Cargo.toml found above CWD)")
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
print(f"CWD: {os.getcwd()}")

# Fail loud upfront if the toolchain is missing — better than a confusing
# error 30 seconds into phase 1.
assert shutil.which("cargo"), "cargo not on PATH — run from inside `nix develop`"
assert shutil.which("java"),  "java not on PATH — needed by `bench gen lubm`"
print("cargo:", shutil.which("cargo"))
print("java: ", shutil.which("java"))


def run_phase(args, label):
    """Run a kermit subprocess with timing and streamed output.

    Used by phases 1, 2, and 3 — each invokes `cargo run --release -- bench …`
    and may take several minutes on a cold build. Output is streamed live to
    the notebook so the cell does not appear hung. We keep a rolling buffer
    of the last 40 lines and surface it on non-zero exit before raising
    CalledProcessError.
    """
    print(f"⏳ {label} - this may take several minutes...")
    start = time.monotonic()
    proc = subprocess.Popen(
        args,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    last_lines: list[str] = []
    for line in proc.stdout:
        line = line.rstrip()
        last_lines.append(line)
        if len(last_lines) > 40:
            last_lines.pop(0)
        print(line)
    rc = proc.wait()
    elapsed = time.monotonic() - start
    if rc != 0:
        raise subprocess.CalledProcessError(rc, args, output="\n".join(last_lines))
    print(f"✅ {label} - done in {elapsed:.1f}s")
    return proc


## Phase 1 — Generate the LUBM-1 dataset

The generator pipeline (`bench gen lubm`) invokes the committed `lubm-uba.jar` to materialise data for one university, applies hardcoded Univ-Bench TBox forward-chaining entailment, and partitions the resulting N-Triples into per-predicate Parquet files.

The output lands in the platform cache (Linux: `~/.cache/kermit/benchmarks/lubm-1-nb-fulltimeline/`). Re-running this cell overwrites the cache subdir from scratch (~30-60s on warm cargo); the imperative `bench gen` path does not short-circuit on a hash check. (The declarative `bench run <name>` path against a workspace YAML does honor a spec_hash check, but that path is not what we use here.)

The tag `nb-fulltimeline` is private to this notebook so it does not collide with any tag you have used in your own workflow.

In [ ]:
run_phase(
    ["cargo", "run", "--release", "--",
     "bench", "gen", "lubm",
     "--scale", "1",
     "--tag", "nb-fulltimeline"],
    "Phase 1: generate LUBM-1 dataset",
)


## Phase 2 — Run the benchmark (time metric)

`bench run` reads the cached benchmark, builds each index structure for every LUBM query, and times the join using Criterion. The results land in:

- `bench-runs/lubm-demo-time.json` — the machine-readable `BenchReport` (one entry per query × DS × algo).
- `target/criterion/<group>/<dir>/{base,new}/` — the per-iter Criterion artefacts that `kl.load` will read for plotting.

`-i all` expands to the full `IndexStructure` enum (all three structures: `TreeTrie`, `ColumnTrie`, `HashTrie`); `-a leapfrog-triejoin` pins the algorithm. **Caveat:** the cross-product loop does not skip the invalid `(HashTrie, LeapfrogTriejoin)` pairing — it still reaches the `HashTrie` arm, runs `hash_join`, and emits a third report mislabeled `algorithm=LeapfrogTriejoin`. So the only *truthfully* labeled results here are `TreeTrie` and `ColumnTrie` under LeapfrogTriejoin; the plots below group/filter on `data_structure`, so the spurious row surfaces as an extra `HashTrie` series rather than corrupting them. To avoid it entirely, pin each structure in a separate invocation (as `07_lubm_reference_comparison.ipynb` does). `--metrics iteration` restricts measurement to the join phase (skipping `insertion` and per-relation `space` measurements that kermit would otherwise run by default). The Criterion overrides (`--sample-size 10`, `--measurement-time 1`, `--warm-up-time 1`) bring each benchmark group down to ~3-4s — adequate for a teaching demo, not a thesis-grade run.

In [ ]:
run_phase(
    ["cargo", "run", "--release", "--",
     "bench",
     "--sample-size", "10",
     "--measurement-time", "1",
     "--warm-up-time", "1",
     "--report-json", "bench-runs/lubm-demo-time.json",
     "run", "lubm-1-nb-fulltimeline",
     "-i", "all",
     "-a", "leapfrog-triejoin",
     "--metrics", "iteration"],
    "Phase 2: bench run (time)",
)


## Phase 3 — Run the benchmark (space metric)

`--metrics space` swaps Criterion's default time `Measurement` for a custom `SpaceMeasurement` that records `heap_size_bytes()` per iter against the pre-built relation. Because Criterion only supports one `Measurement` per invocation, this must be a separate `bench run` from phase 2.

The reports land in `bench-runs/lubm-demo-space.json` alongside their time-metric counterparts. `kl.load` will discover both via the glob `bench-runs/lubm-demo-*.json` and merge them into a single tidy DataFrame.

In [ ]:
run_phase(
    ["cargo", "run", "--release", "--",
     "bench",
     "--sample-size", "10",
     "--measurement-time", "1",
     "--warm-up-time", "1",
     "--report-json", "bench-runs/lubm-demo-space.json",
     "run", "lubm-1-nb-fulltimeline",
     "-i", "all",
     "-a", "leapfrog-triejoin",
     "--metrics", "space"],
    "Phase 3: bench run (space)",
)


## Phase 4 — Load reports into a pandas DataFrame

`kl.load(glob, criterion_root=…)` parses every `BenchReport` matching the glob, joins it against the Criterion JSON artefacts under `criterion_root/<group>/<dir>/`, and returns a tidy summary DataFrame: one row per `(configuration, query, metric, phase/relation)`. (Per-iteration rows — with `sample_idx` and `per_iter_ns` — come from the separate `kl.load_samples`, which joins back on `(criterion_group, criterion_function)`.)

Column reference: see [`kermit_lab/frame.py`](../kermit_lab/frame.py) for the include-list and [`docs/specs/bench-report-schema.md`](../../../docs/specs/bench-report-schema.md) for the JSON schema.

In [ ]:
import kermit_lab as kl

kl.apply_style()  # apply once per notebook session
df = kl.load("bench-runs/lubm-demo-*.json", criterion_root="target/criterion")
print(f"{len(df)} rows, {len(df.columns)} columns")
df.head()


## Phase 5 — Plots

Four inline figures, each answering one question:

- **`bar_queries`** — for a chosen `(DS, algo)`, how do the LUBM queries compare in mean time?
- **`bar_time`** — for one query, how do the index structures compare in mean time? (Pick a non-trivial query programmatically.)
- **`bar_space`** — what's the heap footprint of each index structure?
- **`tradeoff`** (inline) — space-vs-time scatter, one point per `(DS, query)`. Built from raw matplotlib because `kl.tradeoff` merges on `source_path`, which is incompatible with our two-report split (Criterion needs one measurement type per `bench run`).

Every plot returns a `matplotlib.figure.Figure` for inline display.

In [ ]:
kl.bar_queries(df, ds=["TreeTrie"], algo=["LeapfrogTriejoin"])


In [ ]:
import re

def pick_query(queries):
    """Pick the lowest-numbered LUBM query that isn\'t Q1 (which is trivial).

    Natural numeric sort (not lexicographic) so q2 wins over q10. Falls back
    to the alphabetic minimum if no query name contains an integer.
    """
    def key(q):
        m = re.search(r"\d+", q)
        return int(m.group()) if m else -1
    candidates = [q for q in queries if key(q) != 1]
    if not candidates:
        return sorted(queries)[0]
    return sorted(candidates, key=key)[0]


q = pick_query(df["query"].unique())
print(f"Plotting bar_time for query: {q}")
kl.bar_time(df, query=q)


In [ ]:
kl.bar_space(df)


In [ ]:
import matplotlib.pyplot as plt

# `kl.tradeoff` merges time + space rows on `source_path`, which assumes both
# metrics live in the same report. Our two-report split (separate `bench run`
# invocations for time and space, because Criterion only supports one
# Measurement per invocation) forces a manual scatter that joins on
# (data_structure, algorithm, query) instead.
time_means = (
    df[df.metric == "time"]
    .groupby(["data_structure", "algorithm", "query"], dropna=False)["mean_ns"]
    .mean()
    .reset_index(name="time_ns")
)
space_means = (
    df[df.metric == "space"]
    .groupby(["data_structure", "algorithm", "query"], dropna=False)["mean_ns"]
    .mean()
    .reset_index(name="space_bytes")
)
merged = time_means.merge(space_means, on=["data_structure", "algorithm", "query"])

fig, ax = plt.subplots()
for (ds, algo), grp in merged.groupby(["data_structure", "algorithm"]):
    ax.scatter(
        grp["space_bytes"],
        grp["time_ns"],
        label=f"{ds} / {algo}",
        color=kl.colour_for("data_structure", str(ds)),
        marker=kl.marker_for("algorithm", str(algo)),
        s=40, edgecolor="black", linewidth=0.5,
    )
ax.set_xscale("log")
ax.set_xlabel("Mean heap size (bytes, log scale)")
ax.set_ylabel("Mean iteration time (ns)")
ax.set_title("Space vs time tradeoff (one point per query × DS)")
ax.legend()
fig


## Where to go next

You now have working `bench-runs/` and `target/criterion/` artefacts. The other seven notebooks pick up from here:

- **`01_quick_start.ipynb`** — terser version of phases 4-5 alone.
- **`02_scaling.ipynb`** — log-log scaling plot across scales (try regenerating with `--scale 2` and re-running).
- **`03_compare_ds.ipynb`** — pairwise DS comparison with `kl.compare()` and bootstrap CI.
- **`04_watdiv_queries.ipynb`** — multi-query bars across WatDiv stress runs.
- **`05_distributions.ipynb`** — violin plot from raw samples + Mann-Whitney U test.
- **`06_ablation.ipynb`** — plotting the `ds_*`/`algo_*` optimization axes.
- **`07_lubm_reference_comparison.ipynb`** — the full three-configuration reference comparison across all three index structures.

To explore other generator backends, try `bench gen watdiv --scale 100 --tag demo` (uses the vendored watdiv binary; see `docs/benchmarks/WATDIV.md`).

To compare DS pairs statistically, use `kl.compare(df, baseline="TreeTrie", target="ColumnTrie")` and `kl.bootstrap_ratio_ci(...)`.

**Caveat:** if you `cargo clean` between phases 3 and 4, the Criterion artefacts vanish and `kl.load` will error with a clear "no such directory" message. Re-run phases 2 and 3.